# NB05 — Feature Extraction Evaluation
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: For articles included by each method, run the structured LLM feature extractor (`src/extraction/extractor.py`) to populate the standardised extraction schema. Compare extraction quality against a human-annotated sample.

**Schema fields**: `document_type`, `modality`, `anatomic_site`, `llm_method`, `llm_model`, `task`, `performance_metric`, `dataset`, `sample_size`, `key_finding`

**Outputs**
- `data/processed/extracted_features_boolean.csv`
- `data/processed/extracted_features_standard_rag.csv`
- `data/processed/extracted_features_agentic_rag.csv`
- `data/processed/extraction_agreement.csv` — LLM vs human agreement on 50-doc sample

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir
load_env()

from pathlib import Path
import pandas as pd

## 1. Load included articles per method

In [ ]:
# Load screening results — take 'include' decisions for each method
boolean_df   = pd.read_csv(data_dir() / 'processed' / 'screening_results_boolean.csv')

# Merge with corpus metadata to get title + abstract
corpus_df = pd.read_csv(data_dir() / 'corpus' / 'corpus_metadata.csv')

included_boolean = boolean_df[boolean_df['decision'] == 'include'].merge(
    corpus_df, on='doc_id', how='left'
)
print(f'Boolean included: {len(included_boolean)}')

## 2. Run feature extractor

In [ ]:
from src.extraction.extractor import FeatureExtractor

prompt_template = Path('../prompts/library/extraction_v1.txt').read_text()
extractor = FeatureExtractor(prompt_template=prompt_template)

features_boolean = extractor.extract_from_dataframe(included_boolean[['doc_id', 'title', 'abstract']])
features_boolean.to_csv(data_dir() / 'processed' / 'extracted_features_boolean.csv', index=False)
print(f'Extracted {len(features_boolean)} records (Boolean)')

## 3. LLM vs human extraction agreement (50-doc sample)

In [ ]:
# Load human annotations (to be created manually)
human_annot_path = data_dir() / 'processed' / 'human_annotations_sample.csv'

if human_annot_path.exists():
    human_df = pd.read_csv(human_annot_path)
    FIELDS = ['llm_method', 'llm_model', 'task', 'document_type', 'modality']
    agreement_rows = []
    for field in FIELDS:
        merged = human_df.merge(features_boolean[['doc_id', field]], on='doc_id', suffixes=('_human', '_llm'))
        exact = (merged[f'{field}_human'] == merged[f'{field}_llm']).mean()
        agreement_rows.append({'field': field, 'exact_match_rate': round(exact, 3)})
    agreement_df = pd.DataFrame(agreement_rows)
    agreement_df.to_csv(data_dir() / 'processed' / 'extraction_agreement.csv', index=False)
    print(agreement_df)
else:
    print('Human annotation file not yet available — skipping agreement calculation.')